## Importing libraries and load environment variables


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API LOADED SUCCESSFULLY")

API LOADED SUCCESSFULLY


## Initialize the GROQ LLM

In [14]:
# import time
# from langchain_groq import ChatGroq

# model=ChatGroq(
#     model="openai/gpt-oss-20b",
#     temperature=0, max_retries=2
# )
# response=model.invoke("Explain RAG in One Sentence.")
# print(response.content)

# for chunk in model.stream("Explain RAG in 50 sentence."):
#     print(chunk.content, end="", flush=True)
#     time.sleep(0.02)


from langchain.chat_models import init_chat_model
model =init_chat_model("openai/gpt-oss-20b", model_provider="groq")


for chunk in model.stream("Explain RAG in 50 sentence."):
    print(chunk.content, end="", flush=True)

Retrieval-Augmented Generation (RAG) is a framework that enhances language model responses by incorporating external knowledge retrieved from a document collection.  
It was introduced to address the limitations of purely generative models that rely solely on their internal parameters for knowledge.  
RAG works by first querying a retriever to fetch a set of relevant documents based on the input prompt.  
The retrieved documents are then fed into a generator, which conditions its output on both the prompt and the external text.  
This two-step process allows the model to access up-to-date or domain‑specific information that may not be encoded in its weights.  
The retriever component can be a dense embedding model, a BM25 index, or any other search mechanism.  
In practice, dense retrievers such as DPR or Sentence‑BERT are popular because they capture semantic similarity.  
BM25, a bag‑of‑words ranking algorithm, remains useful for its speed and simplicity.  
The generator is typically

## Load the DOCX document


In [40]:
from langchain_community.document_loaders import Docx2txtLoader

file_path="../data/company_info.docx"

loader=Docx2txtLoader(file_path)

documents=loader.load()

print("Number of documents:", len(documents))
print(documents[0].page_content[:1000])

Number of documents: 1
Project: Banking Application

The Banking Application is a backend-based financial application developed to manage customer accounts and banking operations.

Technology Stack

The backend is developed using Java and Spring Boot.

The application uses PostgreSQL as the primary database.

The application provides REST APIs for account management and banking operations.

The project uses Spring Data JPA for database interaction and Hibernate as the ORM framework.

Maven is used for dependency management and project build.

Git is used for source code management.

Account Management

The application allows customers to:

Create a bank account

View account details

Update customer information

Check account balance

Close an account

Search accounts using customer or account information

Each bank account has a unique account number associated with a customer.

Transaction Management

The application supports basic banking transactions such as:

Money deposit

Money 

## Split the document into chunks

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_Splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30
)

chunks=text_Splitter.split_documents(documents)

print("Number of chunks: ", len(chunks))
for i, chunk in enumerate(chunks[:3]):
    print(f"\nchunk {i+1}:")
    print(chunk.page_content[:300])

Number of chunks:  30

chunk 1:
Project: Banking Application

The Banking Application is a backend-based financial application developed to manage customer accounts and banking operations.

Technology Stack

chunk 2:
Technology Stack

The backend is developed using Java and Spring Boot.

The application uses PostgreSQL as the primary database.

chunk 3:
The application provides REST APIs for account management and banking operations.

The project uses Spring Data JPA for database interaction and Hibernate as the ORM framework.


## Configure Google embeddings


In [47]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
test_vector=embeddings.embed_query("What is Spring Boot?")
print("Embedding dimensions: ", len(test_vector))

Embedding dimensions:  3072


## Create the FAISS vectore store

In [48]:
from langchain_community.vectorstores import FAISS

vectorstore=FAISS.from_documents(documents=chunks, embedding=embeddings)

print("FAISS vector store created successfully")

FAISS vector store created successfully


## Save the vector store locally

In [49]:
vectorstore.save_local("../vectorestore/company_faiss")
print("FAISS vectore store saved")

FAISS vectore store saved


## Test document retrieval

In [50]:
retriever=vectorstore.as_retriever(
    search_kwargs={"k":2}
)

query="Which technology is used in the backend?"

retriever_docs=retriever.invoke(query)

for i, doc in enumerate(retriever_docs):
    print(f"\n Document {i+1}")
    print(doc.page_content)


 Document 1
Technology Stack

The backend is developed using Java and Spring Boot.

The application uses PostgreSQL as the primary database.

 Document 2
Project: Banking Application

The Banking Application is a backend-based financial application developed to manage customer accounts and banking operations.

Technology Stack


## Create the RAG search tool

In [53]:
from langchain.tools import tool
@tool
def search_document(query: str)->str:
    """Search the company DOCX docummeys for relevent information. Use this tool
    when the user asks about information that may be present in the uploaded document."""

    docs=retriever.invoke(query)

    if not docs:
        return "No relevant information found in the document."

    results=[]

    for i, doc in enumerate(docs):
        results.append(
            f"Document chunk {i+1}: \n{doc.page_content}"
        )

    return "\n\n".join(results)

In [56]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the information
provided in the document context.

If the answer is not present in the context, say:
"I could not find this information in the document."

Keep the answer concise.

Document context:
{context}

Question:
{question}
""")

## Test the tool

In [64]:
question = "Which technology is used in the backend and which database is used?"

context = search_document.invoke(question)

response = (prompt | model).invoke({
    "context": context,
    "question": question
})

print(response.content)

Backend: Java with Spring Boot  
Database: PostgreSQL


In [60]:
result = search_document.invoke(
    {"query": "What database is used in the project?"}
)

print(result)

Document chunk 1: 
Technology Stack

The backend is developed using Java and Spring Boot.

The application uses PostgreSQL as the primary database.

Document chunk 2: 
Architecture

The application follows a layered architecture:

Client

   ↓

REST Controller

   ↓

Service Layer

   ↓

Repository Layer

   ↓

PostgreSQL Database


## Define calculator tool

In [ ]:
@tool
def add(a: float, b: float)-> float:
    """Add two numbers."""
    return a+b

@tool
def subtract(a: float, b: float)-> float:
    """Subtract two numbers."""
    return a-b

@tool
def multiply(a: float, b: float)->float:
    """Multiply two numbers."""
    return a*b

@tool
def divide(a: float, b: float)->float:
    """Divide two numbers."""
    if(b==0):
        return ValueError("Cannot divide by zero")
    return a/b

In [73]:
print(divide.invoke({"a":5, "b":6}))
print(divide.invoke({"a":3,"b":0}))

0.8333333333333334
Cannot divide by zero


In [76]:
from langchain.agents import create_agent

tools=[
    add, subtract, divide, multiply, search_document
]

agent=create_agent(model=model, tools=tools)

print("Langgraph agent created successfully")

Langgraph agent created successfully


## Create a helper function


In [79]:
def ask_agent(question: str):
    result=agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":question
            }
        ]
    })

    return result["messages"][-1].content

## Test mathematical operations

In [82]:
answer=ask_agent("Calculate 25 multiplied by 4 and then add 100")

print(answer)

25 multiplied by 4 is 100, and adding 100 gives **200**.


## Test the RAG agent

In [83]:
answer=ask_agent("Which technology is used in the backend Project? and which is the database used")

print(answer)

**Backend Technology**  
- The backend is built with **Java** using the **Spring Boot** framework.

**Database**  
- The application stores its data in **PostgreSQL**.


In [85]:
answer=ask_agent("According to the document, how many backend services "
    "are there? If each service costs 200, calculate the "
    "total estimated cost.")

print(answer)

I’ve searched the document for any explicit mention of a count of backend services, but the text does not list a specific number. It describes a single “backend” built with Java/Spring‑Boot, a layered architecture (Controller → Service → Repository), and a PostgreSQL database, but it does not break the backend into separate micro‑services or enumerate them.

**So, according to the document, the number of backend services is not specified.**

If you were to assume the simplest interpretation—that there is just one backend service—then the cost calculation would be:

- Number of services: 1  
- Cost per service: $200  
- **Total estimated cost = 1 × $200 = $200**

If, in reality, there are more services (e.g., separate services for accounts, payments, notifications, etc.), you would need to count those from a more detailed source or a different part of the documentation.


## Inspect the execution

In [87]:
def debug_agent(question: str):
    result=agent.invoke({
        "messages":[
            {
                "role":"user",
                "content":question
            }
        ]
    })

    for message in result["messages"]:
        print("\n---")
        print("Message type: ", type(message).__name__)
        print("Content:", message.content)

        if getattr(message, "too;_calls", None):
            print("Tool calls:", message.tool_calls)

    return result

In [ ]:
debug_agent("What is 20 multiply by 5?")


---
Message type:  HumanMessage
Content: What is 20 multiply by 5?

---
Message type:  AIMessage
Content: 

---
Message type:  ToolMessage
Content: 100.0

---
Message type:  AIMessage
Content: 20 multiplied by 5 equals **100**.


{'messages': [HumanMessage(content='What is 20 multiply by 5?', additional_kwargs={}, response_metadata={}, id='72edf669-c2c1-43de-8d9a-6121795f134c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What is 20 multiply by 5?" We can compute directly: 20 * 5 = 100. Use the multiply function.', 'tool_calls': [{'id': 'fc_c33b498e-3502-4abb-879d-77fd16e419c0', 'function': {'arguments': '{"a":20,"b":5}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 244, 'total_tokens': 303, 'completion_time': 0.062973891, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.021351732, 'prompt_tokens_details': None, 'queue_time': 0.315656723, 'total_time': 0.084325623}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0f373-26ea-7771-abe2-2ce